# 03 — Statistical Association

EDA suggested several behavioral signals. This notebook tests their association with the eventual default outcome.

The analysis uses:

- point-biserial correlation for continuous variables
- chi-square tests for categorical variables
- Cramér's V as an effect-size measure
- adjusted Pearson residuals for category-level inspection

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pointbiserialr, chi2_contingency

DATA_PATH = Path("../data/synthetic/synthetic_early_modeling_base.csv")
df = pd.read_csv(DATA_PATH)

frequency_days = {
    "Weekly": 7,
    "Bi-weekly": 14,
    "Monthly": 28,
}

df["pass_due_cycle_ratio"] = (
    df["early_max_overdue_days"]
    / df["frequency_name"].map(frequency_days)
)

df["has_consecutive_miss"] = (
    df["early_max_consecutive_missed"] >= 2
).astype(int)

## Continuous variables

In [ ]:
candidate_numeric = [
    "early_missed_installment_count",
    "early_max_consecutive_missed",
    "early_max_overdue_days",
    "early_total_overdue_days",
    "early_recovery_delay_cycles",
    "pass_due_cycle_ratio",
    "overdue_proportion",
    "missed_installment_proportion",
]

results = []

for col in candidate_numeric:
    tmp = df[[col, "is_good_or_bad"]].dropna()

    r, p = pointbiserialr(
        tmp["is_good_or_bad"].astype(int),
        tmp[col]
    )

    results.append({
        "feature": col,
        "r": r,
        "p_value": p,
        "n": len(tmp),
        "abs_r": abs(r)
    })

point_biserial_results = (
    pd.DataFrame(results)
    .sort_values("abs_r", ascending=False)
    .reset_index(drop=True)
)

point_biserial_results

In [ ]:
plot_df = point_biserial_results.sort_values("r")

plt.figure(figsize=(8.5, 5))
plt.barh(plot_df["feature"], plot_df["r"])
plt.axvline(0, linewidth=1)
plt.xlabel("Point-biserial correlation")
plt.ylabel("Feature")
plt.title("Association with eventual default")
plt.tight_layout()
plt.show()

Large samples can produce very small p-values even when the association is modest, so both **effect size and significance** are considered.

## Categorical variables

In [ ]:
categorical_features = [
    "frequency_name",
    "product_group",
    "sector",
    "region",
]

chi_rows = []

for col in categorical_features:
    table = pd.crosstab(df[col], df["is_good_or_bad"])

    chi2, p, dof, expected = chi2_contingency(table)

    n = table.to_numpy().sum()
    r, k = table.shape

    cramers_v = np.sqrt(
        (chi2 / n) / max(1, min(k - 1, r - 1))
    )

    chi_rows.append({
        "feature": col,
        "chi2": chi2,
        "df": dof,
        "p_value": p,
        "cramers_v": cramers_v,
    })

chi_square_results = (
    pd.DataFrame(chi_rows)
    .sort_values("cramers_v", ascending=False)
)

chi_square_results

## Category-level inspection

In [ ]:
def category_default_rates(data, feature):
    out = (
        data.groupby(feature, observed=True)["is_good_or_bad"]
            .agg(["count", "mean"])
            .rename(columns={"mean": "default_rate"})
            .reset_index()
    )
    out["default_rate_pct"] = out["default_rate"] * 100
    return out.sort_values("default_rate_pct", ascending=False)

for feature in categorical_features:
    print(f"\n--- {feature} ---")
    display(category_default_rates(df, feature))

## Adjusted Pearson residuals

In [ ]:
def adjusted_pearson_residuals(table):
    observed = table.to_numpy(dtype=float)

    row_totals = observed.sum(axis=1, keepdims=True)
    col_totals = observed.sum(axis=0, keepdims=True)
    total = observed.sum()

    expected = row_totals @ col_totals / total

    row_prop = row_totals / total
    col_prop = col_totals / total

    pearson = (observed - expected) / np.sqrt(expected)

    adjusted = pearson / np.sqrt(
        (1 - row_prop) @ (1 - col_prop)
    )

    return pd.DataFrame(
        adjusted,
        index=table.index,
        columns=table.columns
    )

for feature in categorical_features:
    table = pd.crosstab(df[feature], df["is_good_or_bad"])
    print(f"\n--- {feature} ---")
    display(adjusted_pearson_residuals(table).round(2))

## Interpretation

Statistical association is one input into the next stage.

A variable is not automatically retained because its p-value is small. It also needs to be:

- available at the prediction point;
- analytically defensible;
- sufficiently interpretable;
- not obviously redundant or leakage-prone.

The next stage therefore moves from **association → representation → feature engineering**.